In [1]:
# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path


from config import Config
import data

In [4]:
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/cached_multigraph/era5_swot.yml")
# cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/multigraph/test.yml")
cfg.quiet = False

In [5]:
mgr = data.DynamicCacheManager(cfg)
cache_dir = mgr.create_cache('train', False)

Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache/8df8b7697db1f3d1/<subset>


Loading basins: 100%|██████████| 515/515 [00:00<00:00, 6804.35it/s]

Wrote 0 new basin files to cache.
✅ Cached resources are loaded and ready.


In [6]:
cache_dir

PosixPath('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache/8df8b7697db1f3d1/train')

In [7]:
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, 'train')

Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!


Building sampling index: 100%|██████████| 515/515 [02:03<00:00,  4.19it/s]


In [26]:
cfg.num_workers = 16
dl = data.CachedBasinGraphDataLoader(cfg, dataset)

Dataloader using 16 parallel CPU worker(s).


In [27]:
import objgraph
import psutil
import os
import gc
from tqdm import tqdm

process = psutil.Process(os.getpid())

gc.collect()
before_objs = objgraph.typestats()


def get_mem_mb():
    return process.memory_info().rss / 1024**2  # MB
    
mem_log = []


count = 0
for basin, date, batch in tqdm(dl):
    mem_log.append(get_mem_mb())
    count +=1
    if count == 100:
        break

gc.collect()
after_objs = objgraph.typestats()

for k in after_objs:
    delta = after_objs[k] - before_objs.get(k, 0)
    if delta > 0:
        print(f"{k}: +{delta}")

import matplotlib.pyplot as plt
plt.plot(mem_log)
plt.xlabel("Iteration")
plt.ylabel("Memory (MB)")
plt.title("Incremental memory usage")
plt.show()

  0%|          | 2/1248 [03:36<37:22:53, 108.00s/it]Exception ignored in: <function _xla_gc_callback at 0x7f1413ffc180>
Traceback (most recent call last):
  File "/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback

    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 

KeyboardInterrupt

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x746148c7f2d0>>
Traceback (most recent call last):
  File "/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

In [16]:
import objgraph, gc

gc.collect()
before_objs = objgraph.typestats()

loop()

gc.collect()
after_objs = objgraph.typestats()

for k in after_objs:
    delta = after_objs[k] - before_objs.get(k, 0)
    if delta > 0:
        print(f"{k}: +{delta}")


100%|██████████| 500/500 [04:00<00:00,  2.08it/s]


list: +3
dict: +1
tuple: +1
method: +5
function: +1
ReferenceType: +1
frozenset: +1
partial: +3
Context: +3
list_iterator: +1
Handle: +1
HistoryOutput: +1
TimerHandle: +2
StringIO: +1


In [14]:
 dataset[0]

peak memory: 3785.09 MiB, increment: 0.00 MiB


In [8]:
cfg.num_workers = 12
dl = data.CachedBasinGraphDataLoader(cfg, dataset)

for basin, date, batch in tqdm(dl):
    pass

Dataloader using 12 parallel CPU worker(s).


  7%|▋         | 89/1248 [08:32<1:51:16,  5.76s/it]
Traceback (most recent call last):
  File "/home/tlanghorst_umass_edu/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/home/tlanghorst_umass_edu/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/multiprocessing/util.py", line 363, in _exit_function
    _run_finalizers()
  File "/home/tlanghorst_umass_edu/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/home/tlanghorst_umass_edu/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tlanghorst_umass_edu/.local/share/uv/python/cpython-3.11.12-linux-x86_64-gnu/lib/python3.11/m

In [21]:
len(dataset.sample_list)

183345

In [7]:
dataset[len(dataset.sample_list)-1]

IndexError: list index out of range

In [20]:
index = 0

basin_id, time_idx =index_map[index]

basin_array = cache_root[basin_id]
dynamic_data = basin_array[time_idx] # Slices one time step

dynamic_data.shape

(1827, 1410)

In [6]:
cache_dir = Path('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache/8df8b7697db1f3d1/test')
cache_dir

PosixPath('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache/8df8b7697db1f3d1/test')

In [8]:
import xarray as xr
idx = len(dataset)-1
basin, end_date = dataset.sample_list[idx]
start_date = end_date - pd.Timedelta(days=cfg.sequence_length - 1)
date_slice = slice(start_date, end_date)

ds = xr.open_zarr(dataset.cache_dir / 'ABOM-28501010', consolidated=False)
ds_slice = ds.sel(date=date_slice)

# NaN padding
# Identify all dynamic and target columns that are required for this sample.
all_dynamic_cols = [col for cols in dataset.features.dynamic.values() for col in cols]
all_needed_cols = set(all_dynamic_cols + dataset.target)
missing_cols = all_needed_cols - set(ds.data_vars)
# If any columns are missing, create NaN placeholders and assign them to the dataset
# We do it here, instead of during data loading, because we only need to pad this slice.
if missing_cols:
    nan_arrays_to_add = {}
    for col in missing_cols:
        nan_da = xr.DataArray(
            data=np.full((len(ds.date), len(ds.subbasin)), np.nan, dtype=np.float32),
            coords={"date": ds.date, "subbasin": ds.subbasin},
            dims=["date", "subbasin"],
        )
        nan_arrays_to_add[col] = nan_da
    ds_slice = ds_slice.assign(**nan_arrays_to_add)




for source, columns in dataset.features.dynamic.items():
    print(source)
    source_cols = [c for c in columns if c in ds_slice.data_vars]
    print(ds_slice[columns])

era5
<xarray.Dataset> Size: 8kB
Dimensions:    (date: 90, subbasin: 1)
Coordinates:
  * date       (date) datetime64[ns] 720B 2024-10-03 2024-10-04 ... 2024-12-31
  * subbasin   (subbasin) object 8B 'ABOM-28501010'
Data variables:
    ssrd_mean  (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    strd_mean  (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    ro_mean    (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    sro_mean   (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    e_mean     (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    tp_mean    (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    sp_mean    (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    d2m_mean   (date, subbasin) float64 720B dask.array<chunksize=(63, 1), meta=np.ndarray>
    t2m_mean   (date, subbasin) f

era5
['ssrd_mean', 'strd_mean', 'ro_mean', 'sro_mean', 'e_mean', 'tp_mean', 'sp_mean', 'd2m_mean', 't2m_mean', 'fal_mean']
swot-river
['d_wse_river', 'wse_u_river', 'wse_r_u_river', 'slope_river', 'slope_u_river', 'slope_r_u_river', 'd_width_river', 'width_u_river', 'area_total_river', 'area_tot_u_river', 'xtrk_dist_river', 'reach_q_river', 'reach_q_b_river', 'obs_frac_n_river', 'xovr_cal_q_river']


In [ ]:
self.index_map = []
basins = sorted(list(self.cache_root.keys()))
for basin_id in basins:
    num_timesteps = self.cache_root[basin_id].shape[0]
    for time_idx in range(num_timesteps):
        self.index_map.append((basin_id, time_idx))

In [6]:
dataset.x_s

<xarray.Dataset> Size: 8MB
Dimensions:          (subbasin: 5238)
Coordinates:
  * subbasin         (subbasin) object 42kB '21005508' ... 'USGS-50037000'
Data variables: (12/188)
    unitarea         (subbasin) float64 42kB -1.204 -1.021 ... -0.3202 -1.004
    reservoir        (subbasin) float64 42kB -0.1413 -0.1413 ... -0.1413 -0.1413
    total_area       (subbasin) float64 42kB -0.1592 -0.1613 ... -0.1762 -0.1756
    dis_m3_pyr       (subbasin) float32 21kB -0.1109 -0.1109 ... -0.17 -0.1579
    dis_m3_pmn       (subbasin) float32 21kB -0.1245 -0.1245 ... -0.1411 -0.1331
    dis_m3_pmx       (subbasin) float32 21kB -0.115 -0.115 ... -0.1885 -0.1746
    ...               ...
    hft_ix_s09       (subbasin) float64 42kB 1.294 1.294 1.421 ... 2.754 1.699
    gad_id_smj       (subbasin) float64 42kB -0.525 -0.525 ... 0.481 0.481
    gdp_ud_sav       (subbasin) float64 42kB -1.345 -1.345 ... -1.304 -1.304
    gdp_ud_ssu       (subbasin) float32 21kB 0.1845 0.1845 ... 1.16 0.8439
    hdi_ix_sav       (subbasin) float64 42kB 0.08698 0.08698 ... -2.243 -2.243
    hybas_area_diff  (subbasin) float64 42kB -0.1724 0.05349 ... 0.01348 0.5106